## Formatting for Plots

In [ ]:
# Set import path for custom libraries
import sys
from pathlib import Path
root = Path().resolve()
src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from analysislib import formatting

# Format the notebook
formatting.format_notebook()

---

## Part 1 — Introduction

---

<br>

<div style="width: 1200px">
<center>
<span style="
border: 2px solid #DDDDDD;
padding: 15px;
font-size: 20px;
font-style: italic;">
"What drives galaxy transformation, and can we predict where a galaxy is in its evolutionary lifecycle?"</span>
</center>

<br>

This is the central question guiding this entire project. To answer it, we need real data, and lots of it. In this introduction section, we will meet the dataset that is used for the data analysis in this project: the **Sloan Digital Sky Survey (SDSS)**.

However, before we touch a single number, we'll need build up a clear picture of what the SDSS is, how its data was collected, and what each column in the dataset actually represents physically.

By the end of this notebook, you will:

* Understand how the SDSS telescope captures data about the sky
* Know what a spectrograph does and why redshift matters
* Be able to distinguish between stars, galaxies, and quasars
* Have taken a first look at the dataset's shape, contents, and quality
* Have cleaned the data so it is ready for analysis in later chapters

<br>

<center><img width=500px src="images/hst_galaxies.jpg"></img>

<i>Credit: ESA/Hubble & NASA, F. Pacaud, D. Coe</i></center>
</div>

---

### Assumed Background Knowledge

---

This project is aimed at a general audience, but a little background knowledge will help. Specifically, you should be comfortable with:

* The idea that light behaves like a wave with a **wavelength**, and that wavelength determines colour
* Basic astronomical terms like "star" and "galaxy"
* General scientific / maths vocabulary like: "telescope", "graph", "machine learning", "mean", "median" and "mode"

Everything else will be introduced and explained along the way :)

---

### The Sloan Digital Sky Survey

---

<div style="
display: inline-block;
width: 650px;
margin-right: 100px">

The **Sloan Digital Sky Survey (SDSS)** is one of the most ambitious astronomical projects ever undertaken. Since it began in 2000, a dedicated telescope in New Mexico has been systematically photographing and measuring large portions of the night sky, producing a publicly available dataset that's now the foundation of thousands of scientific studies worldwide.

The thing that makes the SDSS so powerful is that it uses both **photometry** and **spectroscopy**, which together give us both a broad overview of the sky and a deeper understanding of the individual objects.

For this project, we use the first 500,000 entries of the SDSS-V Data Release 19 (July 2025). This sample gives us enough objects to find some statistical patterns in galaxy properties, which will ultimately let us figure out where galaxies are in their evolutionary lifecycle.

</div>

<div style="
display: inline-block;
width: 400px;">
<img src="images/sdss_telescope.jpg"></img>
<center><i>Credit: Patrick Gaulme</i></center>
</div>

---

### How the SDSS Telescope Sees the Sky

---

The SDSS collects data in two very different ways

#### Photometry

<div style="width: 1200px">

**Photometry** is the process of measuring how bright an object appears across different wavelengths of light.

Rather than taking a single image, the SDSS observes each object through five filters labelled `u`, `g`, `r`, `i`, and `z`, which cover a range from ultraviolet to infrared. Each filter captures only a small part of the spectrum, and the combination of all five tells us the object's overall **colour profile**.

Colour turns out to be deeply informative:

* **Blue galaxies** tend to be hot and actively forming new stars
* **Red galaxies** are usually older, cooler, or have stopped forming stars

<br>

<center>
<div style="width: 800px">
<img src="images/em_spectrum.png"></img>
</div>
</center>

<center><i>Credit: Haylam Yuen</i></center>
</div>

#### Spectroscopy

<div style="
display: inline-block;
width: 450px;
margin-right: 25px">

While photometry gives us colour and shape, **spectroscopy** tells us more complex details about the objects.

A **spectrograph** works by taking the incoming light from a celestial object and fanning it out into a full spectrum. But unlike a rainbow, this spectrum contains sharp **absorption and emission lines** at very specific wavelengths.

Interestingly, every chemical element interacts with light at its own unique set of wavelengths, a bit like a fingerprint. By identifying which lines appear in a spectrum, astronomers can determine **exactly which elements are present** in the object.

Spectroscopy can therefore tell us:

* The **chemical composition** of the object (hydrogen, helium, heavier metals)
* The **type** of object and its **age**
* And crucially, its **redshift**, which tells us how far away it is (more on this below)

</div>

<div style="
display: inline-block;
width: 700px;">
<img src="images/spectra_emission.png"></img>

<center><i>Credit: NASA</i></center>
</div>

---

### The Idea of Redshift

---

<div style="width: 1200px">

<center>
<img src="images/redshift_diagram.jpg" width=550px></img>

<i>Credit: Bartleby</i>
</center>

<br>

One of the most important quantities in this dataset is **redshift**. To understand it intuitively, imagine light as a series of waves rippling outward from a distant object. As the universe expands, space itself is being stretched, and as it stretches, so do the light waves travelling through it.

Because of this, light from distant objects arrives at Earth with a longer wavelength than it had when it was emitted. Since longer wavelengths sit toward the red end of the visible spectrum, we call this **redshift**. Conversely, objects moving toward us have their light compressed to shorter (bluer) wavelengths. This is called **blueshift**, although this is rare on cosmological scales.

Redshift is powerful for two main reasons:

1. It lets us figure out **Distance**: A higher redshift generally means the object is farther away.

2. It tells us about the **Universe's past**: Because light travels at a finite speed (~300,000 km/s), observing a galaxy with high redshift means that we get to look into the past. For example, if a galaxy has z = 0.5, we are seeing it as it was about 5 billion years ago (which is awesome!)

<br>

The interactive plot below lets you simulate how redshift shifts the colour of objects. The H-alpha emission line (the bright red line hydrogen emits at 656 nm) is a real reference used by astronomers to measure galaxy distances:

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

wavelength = np.linspace(400, 1000, 1000)

def gaussian(x, mu, sigma):
    return np.exp(-(x - mu)**2 / (2 * sigma**2))

original_line_center = 656.28  # H-alpha line in nm
original_spectrum = gaussian(wavelength, original_line_center, 6)

def plot_redshift(z):
    fig, ax = plt.subplots(figsize=(10, 5), facecolor="#1e1e1e")
    ax.set_facecolor("#1e1e1e")

    shifted_center = original_line_center * (1 + z)
    shifted_wavelength = wavelength * (1 + z)

    ax.plot(wavelength, original_spectrum, "--", linewidth=1.5, alpha=0.3, color="#4dabf7",
            label=f"Original H-α line ({original_line_center:.1f} nm)")
    ax.plot(shifted_wavelength, original_spectrum, linewidth=3, color="#ff7043",
            label=f"Redshifted (z = {z:.2f}) → {shifted_center:.1f} nm")
    ax.fill_between(shifted_wavelength, original_spectrum, color="#ff7043", alpha=0.1)

    ax.text(shifted_center, 1.12, f"{shifted_center:.1f} nm", color="#ff7043",
            ha="center", fontweight="bold", fontsize=10)

    ax.set_xlabel("Wavelength (nm)", color="#cccccc")
    ax.set_ylabel("Relative Intensity", color="#cccccc")
    ax.set_title("Interactive Redshift Simulator — H-alpha Spectral Line",
                 color="#ffffff", loc="left", pad=20)
    ax.tick_params(colors="#cccccc")
    ax.grid(True, linestyle=":", alpha=0.2, color="#ffffff")
    ax.set_xlim(300, 1800)
    ax.set_ylim(0, 1.35)
    for spine in ax.spines.values():
        spine.set_edgecolor("#444444")
    ax.legend(facecolor="#1e1e1e", edgecolor="#444444", labelcolor="#e0e0e0")
    plt.tight_layout()
    plt.show()

z_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.5, step=0.01,
    description="Redshift z:",
    continuous_update=True,
    layout=widgets.Layout(width="80%", margin="10px 0"),
    style={"description_width": "initial"}
)

ui = widgets.VBox([z_slider],
                  layout=widgets.Layout(display="flex", align_items="center",
                                        width="100%", padding="20px 0"))
out = widgets.interactive_output(plot_redshift, {"z": z_slider})
display(ui, out)

---

### The Three Types of Objects in the SDSS

---

<div style="width: 1200px">

Every object in the SDSS is assigned to one of three categories:

<br>

**STARS**

Stars are individual suns just like our own. In the SDSS dataset, all detected stars belong to our own galaxy, the Milky Way, making them relatively nearby on cosmic scales. They appear as **point sources** with no visible structure. Stars are not the main subject of this project, but they appear in the data and can be used as calibration references.

<br>

**GALAXIES**

Galaxies are the main stuff we're focused on in this project. Each one is an enormous thing containing anywhere from millions to trillions of stars, plus big clouds of gas, dust, and invisible dark matter. Unlike stars, galaxies can appear **extended**, meaning they have visible shape and structure. They evolve dramatically over cosmic time: merging, quenching their star formation, changing colour and morphology. Tracing this evolution is the main aim of the project.

<br>

**QUASI-STELLAR OBJECTS / QUASARS (QSOs)**

Quasars are some of the brightest objects in the known universe. They are powered by **supermassive black holes** at the centres of galaxies, which are actively consuming surrounding matter and releasing masssive amounts of energy in the process. This energy can outshine the entire host galaxy, making quasars appear **point-like** (similar to stars). Their extreme brightness and high redshifts make them valuable for looking into early universe.

</div>

---

### Bias and Limitations of the SDSS

---

No dataset is perfect, so addressing the SDSS's known limitations is important for doing the best analysis possible.

* **Flux limit (Malmquist bias):** The telescope can only detect objects above a minimum brightness. This means faint, distant galaxies are systematically underrepresented and the sample is biased toward intrinsically bright or nearby objects.

* **Spectroscopic target selection:** Not every photometrically detected object gets a spectroscopic reading. The SDSS prioritised certain object types for this, so the dataset might over-represent large red galaxies and quasars relative to fainter, "normal" objects.

* **Sky coverage:** The SDSS primarily surveys the Northern Galactic Cap, leaving large portions of the sky unmapped. Large-scale structural studies may therefore be incomplete.

* **Fibre collision:** The physical spectrograph uses optical fibres to capture light. These fibres cannot be placed closer than about 55 arcseconds apart (1 60th of a degree), meaning that dense environments like galaxy clusters may be undersampled.

* **Sample ordering:** The first 500,000 entries of the database may not be a truly random draw from the sky. Database ordering (by sky region or observation date) could introduce hidden structure in the sample.

It's important that we keep these caveats in mind throughout the analysis.

---

### First Look at the Data

---

Now that we've addressed some of the important background information of the SDSS, it's time to load the dataset and begin exploring its structure.

In [ ]:
import pandas as pd

df = pd.read_csv("data/SDSS_500k_v3.csv")
print(f"Dataset loaded: {len(df):,} objects, {df.shape[1]} columns")

Let's inspect the first few rows. Each row is one celestial object; each column is a measured property of that object.

In [ ]:
df.head()

And a statistical summary of all numerical columns:

In [ ]:
df.describe()

#### What Do the Columns Mean?

<div style="width: 1200px;">

Here is a guide to every column — the physical meaning behind each number:

| Column | Physical meaning |
|--------|------------------|
| `objID` | Unique identifier from photometric data — use this to look up any object in the SDSS online database |
| `specObjID` | Unique identifier from spectroscopic data |
| `ra`, `dec` | **Right Ascension** and **Declination** — the longitude and latitude of the sky. Together they give an object's precise position on the celestial sphere |
| `u`, `g`, `r`, `i`, `z` | Brightness in the five photometric bands: ultraviolet, green, red, near-infrared, and mid-infrared. Differences between these values encode the object's **colour** |
| `class` | Object type: `GALAXY`, `STAR`, or `QSO` |
| `redshift` | The measured redshift $z$ — a proxy for distance and lookback time |
| `zWarning` | A bitmask flagging issues during spectral redshift fitting. `0` means no problems. Non-zero values indicate the redshift measurement may be unreliable |
| `flags` | A bitmask recording image processing events (e.g. interpolated pixels). Multiple flags are combined by addition |
| `petroRad_r` | **Petrosian radius** in the r-band — a standardised measure of apparent galaxy size |
| `petroMag_r` | **Petrosian magnitude** in the r-band — total brightness within the Petrosian radius |
| `expRad_r`, `deVRad_r` | Effective radii from exponential (disc) and de Vaucouleurs (bulge) profile fits |
| `fracDeV_r` | Fraction of light accounted for by the de Vaucouleurs profile — `1.0` is a pure elliptical, `0.0` is a pure disc |
| `expAB_r`, `deVAB_r` | Axis ratios from the two profile fits — how round vs. elongated the object appears |

<br>

<center>
<img width=250px src="images/ra_dec.jpg"></img>

<i>Right Ascension and Declination. Credit: Bob King</i>
</center>

<br>

<center>
<img width=400px src="images/petrosian_r.jpg"></img>

<i>The Petrosian radius — a robust way to measure galaxy size regardless of distance. Credit: Michael Richmond</i>
</center>

</div>

---

### What Are We Working With? Three Diagnostic Plots

---

Before cleaning anything, let's get a feel for the raw dataset through three diagnostic visualisations.

In [ ]:
# Import necessary libraries
import matplotlib.pyplot as plt

# Setup for plotting
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Pie chart for class distribution
class_counts = df["class"].value_counts()
colours = ["#5588dd", "#dd8855", "#dd5555"]
df["class"].value_counts().plot.pie(autopct="%1.1f%%", ax=axes[0], startangle=90, color=colours)
axes[0].set_title("Object Type Distribution")
axes[0].set_ylabel("")

# Redshift distribution histogram
galaxy_z = df.loc[df["class"] == "GALAXY", "redshift"]
qso_z = df.loc[df["class"] == "QSO", "redshift"]
axes[1].hist(galaxy_z, bins=60, range=(0, 4), color="#5588dd", alpha=0.7, label=f"Galaxies (median z = {galaxy_z.median():.2f})")
axes[1].hist(qso_z, bins=60, range=(0, 4), color="#dd8855", alpha=0.7, label=f"QSOs (median z = {qso_z.median():.2f})")
axes[1].set_xlabel("Redshift (z)")
axes[1].set_ylabel("Count")
axes[1].set_title("Redshift Distribution: Galaxies vs QSOs", fontweight="bold", pad=12)
axes[1].legend()

# Overall title and display
plt.suptitle("First Look at the SDSS Dataset", fontsize=20, fontweight="bold", y=1.05)
plt.show()

Three things stand out immediately:

1. **Galaxies dominate** — they make up the majority of the sample, confirming this dataset is well-suited to galaxy evolution studies.

2. **The sky map shows clear structure** — the SDSS's coverage of the Northern Galactic Cap is visible as distinct stripes and patches. This is not a random sample of the whole sky.

3. **QSOs reach far deeper into the universe than galaxies** — the redshift histograms reveal that quasars extend to much higher $z$ values, reflecting their extreme luminosity. This depth difference is one of the most visually striking features of the dataset.

---

### Cleaning the Data

---

Raw data from telescopes always needs some tidying before it can be trusted for analysis. We'll apply three targeted cleaning steps.

**Step 1 — Remove unreliable redshift measurements**

Any object with a non-zero `zWarning` has a potentially unreliable redshift. Since redshift is central to almost every downstream analysis, we remove these entries.

In [ ]:
n_before = len(df)
df = df[df["zWarning"] == 0]
n_after = len(df)
print(f"Removed {n_before - n_after:,} objects with non-zero zWarning")
print(f"Retained {n_after:,} objects ({100 * n_after / n_before:.1f}% of original)")

**Step 2 — Remove invalid photometry**

Occasionally, one or more of the five photometric bands (`u`, `g`, `r`, `i`, `z`) contains a missing, zero, or negative value — which is physically impossible for a brightness measurement. We keep only objects with valid readings across all bands.

In [ ]:
photometry_bands = ["u", "g", "r", "i", "z"]
n_before = len(df)
for band in photometry_bands:
    df = df[df[band] > 0]
n_after = len(df)
print(f"Removed {n_before - n_after:,} objects with invalid photometry")
print(f"Retained {n_after:,} objects")

**Step 3 — Split into per-class subsets**

For later notebooks, it will be useful to have three separate DataFrames — one for each object class. This makes targeted analysis much cleaner.

In [ ]:
galaxies = df[df["class"] == "GALAXY"].copy()
stars    = df[df["class"] == "STAR"].copy()
qsos     = df[df["class"] == "QSO"].copy()

print(f"Galaxies: {len(galaxies):,}")
print(f"Stars:    {len(stars):,}")
print(f"Quasars:  {len(qsos):,}")

**Step 4 — Remove galaxies with missing structural measurements**

Some galaxies are missing valid values for `petroRad_r` or `fracDeV_r` — the size and morphology metrics we will rely on heavily in later chapters. We keep only galaxies with positive, physically meaningful values for both.

In [ ]:
n_before = len(galaxies)
galaxies = galaxies[(galaxies["petroRad_r"] > 0) & (galaxies["fracDeV_r"] >= 0)]
n_after = len(galaxies)
print(f"Removed {n_before - n_after:,} galaxies with invalid structural measurements")
print(f"Final galaxy sample: {n_after:,} objects")

---

### Mapping the Sky

---

We'll close this introductory notebook with an interactive sky map — a bird's-eye view of everything we'll be working with.

Each point is one celestial object, plotted at its precise sky position (Right Ascension on the x-axis, Declination on the y-axis). Galaxies, stars, and quasars are coloured separately. You can zoom and pan to explore individual regions of the sky.

In [ ]:
import plotly.express as px

df_sample = df.sample(min(len(df), 200000), random_state=42)
hover_cols = {col: True for col in ["objID", "class", "redshift"] if col in df_sample.columns}

fig = px.scatter(
    df_sample,
    x="ra",
    y="dec",
    color="class",
    color_discrete_map={"GALAXY": "#4dabf7", "STAR": "#f28e2b", "QSO": "#e15759"},
    labels={"ra": "Right Ascension (°)", "dec": "Declination (°)", "class": "Object class"},
    title="Sky Map of SDSS Objects (200,000 object sample)",
    hover_data=hover_cols
)

fig.update_traces(marker=dict(size=2, opacity=0.6))

fig.update_layout(
    template=None,
    paper_bgcolor="#1e1e1e",
    plot_bgcolor="#1e1e1e",
    font=dict(family="monospace", size=12, color="#e0e0e0"),
    title_font=dict(family="monospace", size=18, color="#e0e0e0"),
    xaxis=dict(showgrid=True, gridcolor="#444444", zeroline=False,
               title_font=dict(size=13), tickcolor="#cccccc"),
    yaxis=dict(showgrid=True, gridcolor="#444444", zeroline=False,
               title_font=dict(size=13), tickcolor="#cccccc"),
    width=1000, height=620,
    legend=dict(bgcolor="#2a2a2a", bordercolor="#aaaaaa", borderwidth=1,
                font=dict(size=12, color="#e0e0e0"))
)

fig.show()

The map immediately reveals the SDSS's footprint: the survey covers specific swathes of the Northern sky, with visible gaps where the telescope has not yet observed (or where the galactic plane obscures the view). The clustering of galaxies into filaments and voids — the large-scale structure of the universe — is already faintly visible even in this scatter of points.

---

### What's Next?

We now have a clean, well-understood dataset and a clear picture of what we're working with. In the notebooks that follow, we will progressively zoom in on the galaxy population, examining:

* **Colour bimodality** — why galaxies split into two distinct populations
* **Size, shape, and morphology** — how structure relates to evolutionary state
* **Distances and luminosities** — tracing galaxies across cosmic time
* **Machine learning models** — predicting galaxy class, redshift, and lifecycle position

Let's begin.